# W04D04 — RandomizedSearch + Early Stopping
**ML Engineer Journey | Phase 1 — ML Fundamentals | 2 Apr 2026**

---

## Apa yang akan kita pelajari hari ini

| Topik | Masalah yang diselesaikan |
|---|---|
| RandomizedSearchCV | GridSearch terlalu mahal untuk grid besar |
| Distribusi kontinu (scipy.stats) | Bisa sample nilai apapun di range, bukan hanya list diskrit |
| Early Stopping XGBoost | Hentikan training sebelum overfit terjadi |
| CV vs Early Stopping | Kenapa keduanya tidak bisa dipakai bersamaan langsung |

### Hubungan ke materi sebelumnya

```
W04D03 (GridSearchCV)
  └─ Exhaustive search semua kombinasi
  └─ Total model = kombinasi × fold
  └─ Bottleneck: grid besar = tidak praktis
        │
        ▼
W04D04 (RandomizedSearch + Early Stopping)
  └─ Sample n_iter kombinasi secara acak
  └─ Bisa pakai distribusi kontinu
  └─ Early stopping untuk XGBoost n_estimators
```


---
## Cell 1 — Imports & Setup

In [ ]:
import time
import warnings
import numpy as np
import xgboost as xgb
from scipy.stats import randint, uniform
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, GridSearchCV
)
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore", category=UserWarning)

# Load data — konsisten dengan W04D03
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Features: {data.feature_names[:5]} ...")


---
## Bagian 1 — RandomizedSearchCV

### Analogi

GridSearch = mencicipi semua 1.000 restoran di kota satu per satu.  
RandomizedSearch = pilih 50 secara acak dan ambil yang terbaik.

Hasilnya tidak sempurna — tapi **~95% dari waktu** kamu menemukan parameter yang sangat mendekati optimal, dalam sebagian kecil waktu GridSearch.

### Keunggulan utama: distribusi kontinu

GridSearch hanya bisa pakai **list diskrit** — kamu harus pilih dulu nilai mana yang mau dicoba.  
RandomizedSearch bisa sampling dari **distribusi kontinu** — setiap angka di range bisa keluar, termasuk 0.173, 247, 0.891.


### ⚠️ Jebakan #1 — `uniform(loc, scale)`: Parameter kedua BUKAN upper bound

Ini sumber bug paling umum. Banyak orang membaca `uniform(0.6, 0.4)` sebagai "range dari 0.6 sampai 0.4" — itu **salah**.

Signature aslinya: `uniform(loc, scale)` di mana:
- `loc` = nilai **minimum** (start)
- `scale` = **lebar** range, bukan nilai maximum

Jadi `uniform(0.6, 0.4)` = dari 0.6 hingga **(0.6 + 0.4) = 1.0**


In [ ]:
# DEMONSTRASI: uniform(loc, scale) — parameter kedua adalah LEBAR, bukan upper bound

samples = uniform(loc=0.6, scale=0.4).rvs(size=10, random_state=42)

print("uniform(0.6, 0.4) — 10 sample:")
print(np.round(samples, 4))
print(f"Min: {samples.min():.4f}, Max: {samples.max():.4f}")
print()
print("Kesimpulan: range adalah [0.6, 1.0], BUKAN [0.4, 0.6]")
print()

# Perbandingan langsung
print("Perbandingan — mana yang menghasilkan apa:")
print(f"  uniform(0.01, 0.29).rvs(5) = {np.round(uniform(0.01, 0.29).rvs(5, random_state=1), 3)}")
print(f"  → range [0.01, 0.30]")
print(f"  uniform(0.6, 0.4).rvs(5)  = {np.round(uniform(0.6, 0.4).rvs(5, random_state=1), 3)}")
print(f"  → range [0.60, 1.00]")


### ⚠️ Jebakan #2 — `randint(a, b)`: Upper bound EKSKLUSIF

Berbeda dari Python built-in `random.randint(a, b)` yang **inklusif** di kedua sisi,  
`scipy.stats.randint(a, b)` mengikuti konvensi numpy/Python range: **b eksklusif**.

`randint(50, 500)` = integer acak dari 50 hingga **499** — angka 500 tidak pernah keluar.


In [ ]:
# DEMONSTRASI: randint(a, b) — upper bound eksklusif

samples_int = randint(50, 500).rvs(size=1000, random_state=42)

print("randint(50, 500) — statistik dari 1000 sample:")
print(f"  Min: {samples_int.min()}")
print(f"  Max: {samples_int.max()}")
print(f"  Apakah 500 pernah muncul? {500 in samples_int}")
print(f"  Apakah 499 pernah muncul? {499 in samples_int}")
print()
print("Kesimpulan: range adalah [50, 499], nilai 500 tidak pernah keluar.")


---
## Cell 4 — RandomizedSearch pada Random Forest

### ⚠️ Jebakan #3 — Jangan share satu objek estimator antara dua search

Sklearn melakukan internal clone pada estimator sebelum fitting, tapi bergantung pada ini adalah kebiasaan berbahaya. 
Selalu buat instance baru untuk setiap search object — hindari efek samping yang sulit di-debug.


In [ ]:
# RandomizedSearch pada Random Forest
# Menggunakan distribusi kontinu — ini yang tidak bisa dilakukan GridSearch

param_dist_rf = {
    'n_estimators':     randint(50, 500),    # [50, 499]
    'max_depth':        randint(3, 20),      # [3, 19]
    'min_samples_split':randint(2, 20),      # [2, 19]
    'min_samples_leaf': randint(1, 10),      # [1, 9]
    'max_features':     uniform(0.3, 0.7),   # [0.3, 1.0] — fraksi fitur per split
}

# BENAR: instance terpisah — jangan pakai objek yang sama untuk dua search
rf_for_random = RandomForestClassifier(random_state=42, n_jobs=-1)

random_search_rf = RandomizedSearchCV(
    estimator=rf_for_random,
    param_distributions=param_dist_rf,
    n_iter=30,              # coba 30 kombinasi acak — bukan semua
    cv=5,
    scoring='roc_auc',      # konsisten dengan W04D03
    return_train_score=True,# wajib untuk deteksi gap overfit
    random_state=42,        # reproducible sampling
    n_jobs=-1,
    verbose=0
)

start = time.time()
random_search_rf.fit(X_train, y_train)
elapsed = time.time() - start

best_idx = random_search_rf.best_index_
train_auc = random_search_rf.cv_results_['mean_train_score'][best_idx]
val_auc   = random_search_rf.cv_results_['mean_test_score'][best_idx]
gap       = train_auc - val_auc

print(f"Waktu: {elapsed:.1f}s  |  Total model: 30 × 5 = 150")
print(f"Best params: {random_search_rf.best_params_}")
print(f"Train AUC : {train_auc:.4f}")
print(f"Val AUC   : {val_auc:.4f}")
print(f"Gap       : {gap:.4f}  ({'minimal' if gap < 0.02 else 'tolerable' if gap < 0.05 else 'OVERFIT'})")


---
## Cell 5 — Benchmark: GridSearch vs RandomizedSearch

### ⚠️ Jebakan #4 — RandomizedSearch TIDAK selalu lebih cepat di dataset kecil

Di dataset kecil (seperti breast cancer, 569 baris), overhead setup parallelisme RandomizedSearch bisa melebihi keuntungannya.
Keunggulan RandomizedSearch baru signifikan ketika:
- Grid kombinasi sangat besar (ratusan hingga ribuan)
- Dataset besar
- Atau kamu perlu sampling dari distribusi kontinu


In [ ]:
# Benchmark: GridSearch vs RandomizedSearch
# Pakai XGBoost dengan grid yang SAMA ukurannya untuk perbandingan adil

# BENAR: instance TERPISAH untuk setiap search
xgb_for_grid   = xgb.XGBClassifier(eval_metric='auc', random_state=42, n_jobs=1)
xgb_for_random = xgb.XGBClassifier(eval_metric='auc', random_state=42, n_jobs=1)

# ── GridSearch: 3×3×3 = 27 kombinasi (exhaustive)
param_grid = {
    'n_estimators':  [100, 200, 300],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
}

grid_search = GridSearchCV(
    xgb_for_grid, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=0
)
t0 = time.time()
grid_search.fit(X_train, y_train)
t_grid = time.time() - t0

# ── RandomizedSearch: 20 kombinasi (sampled)
param_dist = {
    'n_estimators':  randint(50, 400),
    'max_depth':     randint(3, 10),
    'learning_rate': uniform(0.01, 0.29),
}

random_search = RandomizedSearchCV(
    xgb_for_random, param_dist, n_iter=20, cv=5,
    scoring='roc_auc', random_state=42, n_jobs=-1, verbose=0
)
t0 = time.time()
random_search.fit(X_train, y_train)
t_random = time.time() - t0

# ── Hasil
print(f"{'='*60}")
print(f"{'Metode':<22} {'Waktu':>8} {'Kombinasi':>12} {'Best AUC':>10}")
print(f"{'-'*60}")
print(f"{'GridSearchCV':<22} {t_grid:>7.1f}s {27:>12} {grid_search.best_score_:>10.4f}")
print(f"{'RandomizedSearchCV':<22} {t_random:>7.1f}s {20:>12} {random_search.best_score_:>10.4f}")
print(f"{'='*60}")

# Interpretasi kecepatan yang benar
ratio = t_grid / t_random
if ratio > 1.0:
    print(f"\nRandomizedSearch lebih cepat ({ratio:.1f}x speedup)")
else:
    print(f"\nGridSearch lebih cepat di dataset kecil ini.")
    print(f"RandomizedSearch {1/ratio:.1f}x lebih lambat karena overhead setup parallelisme")
    print(f"melebihi keuntungan di dataset 569 baris.")
    print(f"Keunggulan RandomizedSearch baru terasa di grid besar (500+ kombinasi).")


---
## Bagian 2 — Early Stopping XGBoost

### Analogi

Kamu sedang belajar untuk ujian. Tiap jam temanmu kasih mini-quiz.  
Di jam ke-3 skor mulai naik. Di jam ke-8 skornya stagnan — bahkan mulai turun.

**Early stopping = teman yang bilang: "Kamu sudah 20 quiz berturut-turut tidak ada peningkatan. Berhenti sekarang."**  
Model disimpan dari titik terbaik, bukan dari titik terakhir.

### Mekanisme

XGBoost melatih pohon **secara sequential** (boosting = satu pohon per iterasi).  
Early stopping memantau val score setelah tiap pohon ditambahkan.  
Jika `N` round berturut-turut tidak ada improvement → training berhenti.  
Model yang disimpan = model pada `best_iteration`, bukan iterasi terakhir.

### ⚠️ Jebakan #5 — `eval_set`: item TERAKHIR yang dipantau

Jika kamu pass `eval_set=[(X_tr, y_tr), (X_val, y_val)]`,  
XGBoost memantau item **terakhir** dalam list untuk early stopping.  
Urutan ini penting — jangan terbalik.


### ⚠️ Jebakan #6 — Early stopping butuh val set terpisah, BUKAN test set

Test set adalah data yang seharusnya tidak pernah "dilihat" model selama proses apapun —  
termasuk early stopping. Gunakan sebagian dari training data sebagai validation set.


In [ ]:
# Early Stopping — val set dari training data (BUKAN test set)

# Split X_train → 80% train, 20% val untuk early stopping
# test set tetap tersimpan bersih untuk evaluasi akhir
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"X_tr:  {X_tr.shape}  ← dipakai untuk training")
print(f"X_val: {X_val.shape}  ← dipantau untuk early stopping")
print(f"X_test: {X_test.shape}  ← TIDAK disentuh sampai evaluasi akhir")
print()

xgb_model = xgb.XGBClassifier(
    n_estimators=500,           # batas atas — early stopping yang putuskan kapan berhenti
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    early_stopping_rounds=20,   # berhenti jika 20 round berturut-turut val score tidak naik
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr), (X_val, y_val)],  # item TERAKHIR yang dipantau
    verbose=50
)

print(f"\n{'='*50}")
print(f"best_iteration  : {xgb_model.best_iteration}")
print(f"best_score      : {xgb_model.best_score:.4f}  (val AUC)")
print(f"n_estimators    : {xgb_model.n_estimators}  (batas atas yang kita set)")
print(f"Iterasi dihemat : {xgb_model.n_estimators - xgb_model.best_iteration}")
print(f"{'='*50}")
print()
print("Ketika predict() dipanggil:")
print(f"  → model menggunakan pohon 0 hingga {xgb_model.best_iteration}")
print(f"  → {xgb_model.n_estimators - xgb_model.best_iteration} pohon setelahnya diabaikan")


### ⚠️ Jebakan #7 — `best_iteration` kecil bukan selalu bagus

`best_iteration = 17` pada percobaan kita artinya model overfit **sangat cepat**.  
Ini sinyal bahwa `learning_rate=0.05` terlalu agresif untuk dataset sekecil ini.  
Di real-world dengan dataset lebih besar, `best_iteration` biasanya berkisar 50–300.

**Verifikasi:** coba turunkan `learning_rate` dan lihat apakah `best_iteration` naik.


In [ ]:
# Verifikasi: learning_rate kecil → best_iteration lebih besar

results = []

for lr in [0.05, 0.02, 0.01]:
    m = xgb.XGBClassifier(
        n_estimators=500, learning_rate=lr, max_depth=5,
        eval_metric='auc', early_stopping_rounds=20,
        random_state=42, n_jobs=-1
    )
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    results.append((lr, m.best_iteration, m.best_score))

print(f"{'learning_rate':>14} {'best_iteration':>15} {'best_val_AUC':>13}")
print("-" * 45)
for lr, bi, bs in results:
    print(f"{lr:>14.3f} {bi:>15} {bs:>13.4f}")

print()
print("Semakin kecil learning_rate → semakin banyak iterasi diperlukan")
print("untuk mencapai performa optimal → best_iteration lebih besar.")


---
## Bagian 3 — Konflik CV vs Early Stopping

### Kenapa keduanya tidak bisa langsung dikombinasikan?

**Cross-validation** membutuhkan: satu val set yang konsisten tidak tersedia  
— data dibagi ke N fold, setiap fold punya val set yang berbeda.

**Early stopping** membutuhkan: satu val set yang **tetap sama** selama seluruh training  
— untuk memantau val score per iterasi.

Keduanya saling bertentangan. Ini bukan masalah implementasi, tapi **konflik arsitektur**.

### Solusi praktis yang digunakan di industri

```
Step 1: RandomizedSearchCV
  - n_estimators FIXED (mis. 200)
  - Tune: learning_rate, max_depth, subsample, dll.
  - Dapat: best_params_ tanpa n_estimators optimal

Step 2: Fit ulang dengan best_params + early stopping
  - Pakai best_params_ dari step 1
  - Biarkan early stopping tentukan n_estimators optimal
  - Dapat: model final yang optimal
```


In [ ]:
# SOLUSI INDUSTRI: RandomizedSearch → fit ulang dengan early stopping

# STEP 1: Tune parameter struktural (BUKAN n_estimators)
param_dist_xgb = {
    'learning_rate':    uniform(0.01, 0.29),   # [0.01, 0.30]
    'max_depth':        randint(3, 10),         # [3, 9]
    'subsample':        uniform(0.6, 0.4),      # [0.6, 1.0]
    'colsample_bytree': uniform(0.5, 0.5),      # [0.5, 1.0]
    'min_child_weight': randint(1, 10),         # [1, 9]
}

# n_estimators FIXED — karena early stopping tidak tersedia di dalam CV
xgb_for_cv = xgb.XGBClassifier(
    n_estimators=200,    # fixed, cukup besar sebagai proxy
    eval_metric='auc',
    random_state=42,
    n_jobs=1             # 1 karena RandomizedSearchCV sudah paralel lewat n_jobs=-1
)

random_search_xgb = RandomizedSearchCV(
    xgb_for_cv, param_dist_xgb,
    n_iter=20, cv=5, scoring='roc_auc',
    return_train_score=True, random_state=42, n_jobs=-1, verbose=0
)

random_search_xgb.fit(X_train, y_train)

best_params = random_search_xgb.best_params_
best_idx    = random_search_xgb.best_index_
train_auc   = random_search_xgb.cv_results_['mean_train_score'][best_idx]
val_auc     = random_search_xgb.cv_results_['mean_test_score'][best_idx]
gap         = train_auc - val_auc

print("STEP 1 — RandomizedSearch selesai")
print(f"Best params: {best_params}")
print(f"Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f} | Gap: {gap:.4f}")
print()

# STEP 2: Fit ulang dengan best_params + early stopping
final_model = xgb.XGBClassifier(
    **best_params,
    n_estimators=1000,          # set tinggi — early stopping yang tentukan
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("STEP 2 — Early stopping selesai")
print(f"n_estimators optimal: {final_model.best_iteration}")
print(f"Val AUC terbaik: {final_model.best_score:.4f}")


---
## Cell 9 — Gap Analysis: Deteksi Overfit

Gap = Train AUC - Val AUC

| Gap | Interpretasi | Tindakan |
|---|---|---|
| < 0.02 | Overfit minimal — generalize baik | Tidak perlu tindakan |
| 0.02 – 0.05 | Overfit tolerable | Monitor, pertimbangkan regularisasi |
| > 0.05 | Overfit serius | Tambah regularisasi, kurangi kompleksitas |

### ⚠️ Jebakan #8 — Train AUC = 1.0 tidak selalu masalah

Jika Train AUC = 1.0 tapi Val AUC = 0.99, gap = 0.01 → model masih baik.  
Yang berbahaya adalah Train AUC = 1.0 dengan Val AUC = 0.75 → gap = 0.25 → overfit serius.  
Jangan fokus pada Train AUC saja, selalu lihat **gap-nya**.


In [ ]:
# Gap analysis untuk semua model yang sudah dilatih hari ini

models_gap = [
    ("RF RandomizedSearch",  random_search_rf),
    ("XGBoost RandomizedSearch", random_search_xgb),
]

print(f"{'Model':<28} {'Train AUC':>10} {'Val AUC':>10} {'Gap':>8} {'Status':>12}")
print("-" * 72)

for name, search in models_gap:
    idx  = search.best_index_
    tr   = search.cv_results_['mean_train_score'][idx]
    vl   = search.cv_results_['mean_test_score'][idx]
    gap  = tr - vl
    if gap < 0.02:
        status = "Minimal"
    elif gap < 0.05:
        status = "Tolerable"
    else:
        status = "OVERFIT"
    print(f"{name:<28} {tr:>10.4f} {vl:>10.4f} {gap:>8.4f} {status:>12}")


---
## Ringkasan & Cheatsheet

### Kapan pakai apa?

| Kondisi | Gunakan |
|---|---|
| Grid kecil (< 100 kombinasi), parameter diskrit | GridSearchCV |
| Grid besar, banyak parameter, atau butuh distribusi kontinu | RandomizedSearchCV |
| Tune n_estimators XGBoost secara otomatis | Early Stopping (val set terpisah) |
| Tune parameter lain XGBoost | RandomizedSearch dengan n_estimators fixed |
| Mau keduanya optimal | RandomizedSearch dulu → fit ulang dengan early stopping |

---

### 8 Jebakan Hari Ini — Rangkuman

| # | Jebakan | Yang Benar |
|---|---|---|
| 1 | `uniform(0.6, 0.4)` = range [0.4, 0.6] | `uniform(loc, scale)`: range = [0.6, **1.0**] |
| 2 | `randint(50, 500)` termasuk 500 | Upper bound **eksklusif** — hanya hingga 499 |
| 3 | Share satu objek estimator antara dua search | Buat **instance terpisah** untuk setiap search |
| 4 | RandomizedSearch selalu lebih cepat | Di dataset kecil bisa lebih **lambat** |
| 5 | item pertama `eval_set` yang dipantau | Item **terakhir** yang dipantau untuk early stopping |
| 6 | Pakai test set untuk early stopping | Gunakan val set dari **training data** |
| 7 | `best_iteration` kecil = bagus | Terlalu kecil = learning rate terlalu agresif |
| 8 | Train AUC = 1.0 artinya overfit | Lihat **gap**-nya, bukan Train AUC saja |

---

### Quick Reference Code

```python
# Distribusi yang benar
'learning_rate': uniform(0.01, 0.29),   # range [0.01, 0.30]
'subsample':     uniform(0.6, 0.4),     # range [0.60, 1.00]
'max_depth':     randint(3, 10),        # range [3, 9] — 10 TIDAK termasuk

# Instance terpisah — selalu
xgb_a = xgb.XGBClassifier(...)  # untuk GridSearch
xgb_b = xgb.XGBClassifier(...)  # untuk RandomizedSearch

# eval_set — urutan penting
eval_set=[(X_tr, y_tr), (X_val, y_val)]  # X_val yang dipantau
# BUKAN: eval_set=[(X_val, y_val), (X_tr, y_tr)]  # ini salah — X_tr yang dipantau

# Setelah fit — akses hasil early stopping
xgb_model.best_iteration  # iterasi val score terbaik
xgb_model.best_score       # val score pada best_iteration
xgb_model.predict(X_test)  # otomatis pakai pohon 0 hingga best_iteration
```

---

*github.com/lolivampire/ML-Project | Next: W04D05 — SHAP Values & Explainability*
